```markdown
# Detecção de Objetos em Vídeo com MobileNet-SSD e OpenCV

Este notebook ensina como implementar um sistema de detecção de objetos focado em identificar pessoas em arquivos de vídeo. Utilizando uma rede neural pré-treinada leve e a biblioteca OpenCV, você aprenderá a processar um vídeo frame por frame, aplicar inteligência artificial para localizar objetos e extrair resultados visuais.

## 🎯 Objetivos de Aprendizagem
* Compreender o fluxo básico de processamento de vídeos usando OpenCV.
* Entender o conceito de redes neurais pré-treinadas e o formato Caffe.
* Aprender a transformar imagens em tensores (blobs) para entrada em modelos de Deep Learning.
* Filtrar e interpretar as saídas e caixas delimitadoras (*bounding boxes*) geradas pelo modelo.

## 📚 Pré-requisitos e Bibliotecas Utilizadas
* **`os`**: Biblioteca padrão do Python para interagir com o sistema operacional (manipular caminhos de arquivos).
* **`cv2` (OpenCV)**: A principal biblioteca de visão computacional do mundo, essencial para ler, processar e salvar imagens e vídeos.
* **`numpy`**: Biblioteca de computação científica para manipulação eficiente de matrizes multidimensionais (as imagens e saídas do modelo).
* **`google.colab.patches.cv2_imshow`**: Uma função específica do ambiente Google Colab que substitui o `cv2.imshow` tradicional, evitando travamentos do navegador ao renderizar imagens diretamente no notebook.
```

In [ ]:
import os
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

```markdown
### Passo 2: Definição do Modelo e das Classes de Objetos

**O que este passo faz:**
Define as variáveis de caminho para os arquivos de configuração estrutural do modelo (`MODEL_CONFIG`), os pesos treinados da rede neural (`MODEL_WEIGHTS`) e a lista completa de categorias (`CLASSES`) que este modelo é capaz de reconhecer.

**Por que ele é necessário:**
* **MobileNetSSD_deploy.prototxt**: É o arquivo de configuração que descreve a arquitetura da rede neural (como as camadas de processamento estão conectadas).
* **MobileNetSSD_deploy.caffemodel**: Contém os pesos numéricos que a rede aprendeu durante o processo de treinamento anterior (o "cérebro" propriamente dito).
* **CLASSES**: O modelo não nos devolve o nome "person" ou "car" diretamente, mas sim um índice numérico (ex: 15). Precisamos desse mapeamento de texto ordenado para traduzir o índice no nome da classe correspondente.
```

In [ ]:
MODEL_CONFIG = "MobileNetSSD_deploy.prototxt"
MODEL_WEIGHTS = "MobileNetSSD_deploy.caffemodel"
CLASSES = ["background","aeroplane","bicycle","bird","boat", "bottle", "bus",
           "car", "cat", "chair", "cow", "dinigtable", "dog", "horse", "motorbike", "person",
           "pottedplant","sheep","sofa","train","tvmonitor"]

```markdown
### Passo 3: Carregando a Rede Neural

**O que este passo faz:**
Carrega a rede neural na memória do sistema utilizando o módulo DNN (Deep Neural Network) da biblioteca OpenCV através da função `cv2.dnn.readNetFromCaffe`.

**Por que ele é necessário:**
Apenas ter os arquivos no disco não é suficiente. Precisamos inicializar a estrutura computacional da rede no Python para que ela esteja pronta para receber nossas imagens e realizar predições.

💡 **Para pensar:** Se o arquivo `MobileNetSSD_deploy.prototxt` ou `MobileNetSSD_deploy.caffemodel` não estiverem presentes no diretório atual do Colab, esta linha de código resultará em erro. Seria útil adicionar um passo anterior para baixar esses arquivos automaticamente se não forem encontrados.
```

In [ ]:
net = cv2.dnn.readNetFromCaffe(MODEL_CONFIG,MODEL_WEIGHTS)

```markdown
### Passo 4: Função Principal de Processamento e Detecção

**O que este passo faz:**
Cria uma função (`detect_person_in_video`) que abre um arquivo de vídeo, lê cada imagem (frame) sequencialmente, prepara essa imagem para a inteligência artificial, executa o modelo, desenha caixas de marcação caso detecte uma pessoa, salva o frame e interrompe o vídeo na primeira detecção positiva.

**Conceitos chave:**
* **`cv2.VideoCapture`**: Abre o arquivo de vídeo facilitando o acesso sequencial aos frames.
* **`cv2.dnn.blobFromImage`**: Redes neurais não aceitam imagens puras diretamente de forma eficiente. Esta função converte a imagem para o formato correto (*blob*), redimensionando-a para $300 \times 300$ pixels, aplicando fator de escala e normalizando as cores.
* **`net.forward()`**: Executa a passagem direta (inferência) através da rede neural para obter as predições de objetos.
* **`confidence`**: O grau de certeza (de 0 a 1) que o modelo possui de que o objeto detectado pertence àquela classe.

💡 **Para pensar:** Atualmente, a função encerra o vídeo (`return`) logo no primeiro frame em que uma pessoa é detectada. Se o objetivo for analisar e gerar um vídeo de saída completo, o ideal seria processar todos os frames até o final e salvar o resultado como um novo arquivo de vídeo, em vez de parar no primeiro frame detectado.
```

In [ ]:
def detect_person_in_video(video_path):
  cap = cv2.VideoCapture(video_path)
  frame_count = 0
  while True:
    ret, frame = cap.read()
    if not ret:
      print("Fim do vídeo ou erro de leitura")
      break
    frame_count += 1
    (h,w) = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 0.007843, (300,300), 127.5)
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
      confidence = detections[0,0,i,2]
      if confidence > 0.5:
        idx = int(detections[0,0,i,1])
        if CLASSES[idx] == "person":
          box = detections[0,0,i,3:7] * np.array([w,h,w,h])
          (startX, startY, endX, endY) = box.astype("int")

          label  = f"{CLASSES[idx]}: {confidence * 100:.2f}%"
          cv2.rectangle(frame, (startX,startY),(endX, endY),(0,255,0),2)
          cv2.putText(frame, label, (startX, startY-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5,(0,255,0),2)
          print(f"Pessoa Detectada no Frame {frame_count}")
          cv2_imshow(frame)

          detected_frame_path = f"detected_frame_{frame_count}.jpg"
          cv2.imwrite(detected_frame_path, frame)
          print(f"Frame salvo como {detected_frame_path}")

          cap.release()
          return "Pessoa detectada no vídeo"
  cap.release()
  return "Nenhuma pessoa detectada no vídeo"


```markdown
### Passo 5: Teste do Modelo em um Arquivo de Vídeo

**O que este passo faz:**
Chama a função criada passando como argumento o arquivo de vídeo `

In [ ]:
print(detect_person_in_video("nopeople.mp4"))